In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats


In [ ]:
repo_dir = Path("../../..")

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

from visualization.src.utils import (
    BENCHMARK_NAME_MAPPING,
    BENCHMARK_COLORS,
    ARCHITECUTURE_FAMILY_COLORS,
    apply_hiearchical_aggregation,
    save_figs,
)


In [ ]:
sns.set_theme(style="ticks")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir_supp = repo_dir / "visualization" / "paper" / "supp" / "figures"
save_dir_main = repo_dir / "visualization" / "paper" / "main" / "figures"
# save_dir.mkdir(parents=True, exist_ok=True)

results_file = artifacts_dir / "pretraining_results_with_metadata.csv"
assert results_file.exists(), f"results file {results_file} does not exist!"

df_all = pd.read_csv(results_file, low_memory=False)
df_all.shape


In [ ]:
DPI = 300

# Pretraining Task Accuracy and Neural Alignment

This appendix examines whether classification accuracy acquired during supervised pretraining is associated with neural alignment, measured as noise-corrected Pearson correlation, $r_{\mathrm{NC}}$. Analyses include non-adversarial SPVVS models pretrained on ImageNet or Ecoset; the main summary averages across neural benchmarks, while appendix figures resolve benchmark- and ROI-specific effects and the aggregate high-accuracy comparison.

In [ ]:
ALIGNMENT_METRIC = "pearsonr_nc"
ACCURACY_METRIC = "task_evaluation_acc"

BENCHMARKS = [
    "bs_fz",
    "bs_mh",
    "tvsd",
    "things_fmri",
    "nsd_func1pt8mm_individualROIs",
    "things_eeg1",
    "things_eeg2",
    "things_meg",
]

MODEL_GROUP_COLS = [
    "model_id",
    "backbone_source",
    "backbone_arch",
    "backbone_arch_family",
    "pretraining_dataset",
    "pretraining_samples_per_class",
    "pretraining_seed",
    "backbone_n_params",
    "pretraining_total_flops_estimate",
]

ARCH_FAMILY_ORDER = [
    "AlexNet",
    "CORnet-S",
    "ResNet",
    "EfficientNet",
    "ConvNeXt",
    "ViT",
]

ARCH_FAMILY_MAPPING = {
    "ResNetFlex": "ResNet",
    "ConvNeXtFlex": "ConvNeXt",
    "ViTFlex": "ViT",
}

arch_family_palette = {
    family: ARCHITECUTURE_FAMILY_COLORS.get(family, color)
    for family, color in zip(
        ARCH_FAMILY_ORDER,
        sns.color_palette("tab10", n_colors=len(ARCH_FAMILY_ORDER)).as_hex(),
    )
}

ARCH_FAMILY_MARKERS = {
    "AlexNet": "o",
    "CORnet-S": "s",
    "ResNet": "^",
    "EfficientNet": "D",
    "ConvNeXt": "P",
    "ViT": "X",
}

DATASET_NAME_MAPPING = {
    "imagenet": "ImageNet",
    "ecoset": "Ecoset",
}

DATASET_PALETTE = {
    "ImageNet": "#2b8cbe",
    "Ecoset": "#e34a33",
}

STATS_LINE_SPACING = 0.1
DATASET_COMPARE_STATS_LINE_SPACING = 0.045
PANEL_TITLE_FONTSIZE = 16
AXIS_LABEL_FONTSIZE = 12
SUPTITLE_FONTSIZE = 20
LEGEND_FONTSIZE = 13
ROI_LEGEND_FONTSIZE = 11
LEGEND_MARKER_SIZE = 11

ROI_NAME_MAPPING = {
    "whole_brain": "Whole Brain",
}

ROI_ORDER = {
    "bs_fz": ["V1", "V2"],
    "bs_mh": ["V4", "IT"],
    "tvsd": ["V1", "V4", "IT"],
    "things_eeg1": ["occipital", "parietal", "temporal", "frontal", "central", "occipital_parietal", "Whole Brain"],
    "things_eeg2": ["occipital", "parietal", "temporal", "frontal", "central", "occipital_parietal", "Whole Brain"],
    "things_meg": ["occipital", "parietal", "temporal", "frontal", "central", "occipital_parietal", "Whole Brain"],
    "things_fmri": [
        "V1", "V2", "V3", "V4", "VO1", "VO2", "LO1", "LO2", "TO1", "TO2", "V3b", "V3a",
        "lEBA", "rEBA", "lFFA", "rFFA", "lOFA", "rOFA", "lPPA", "rPPA", "lRSC", "rRSC", "lTOS", "rTOS", "lLOC", "rLOC", "IT", "Whole Brain",
    ],
    "nsd_func1pt8mm_individualROIs": [
        "early", "midventral", "midlateral", "midparietal", "ventral", "lateral", "parietal", "V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4",
        "nsdgeneral", "OWFA", "VWFA-1", "VWFA-2", "OPA", "PPA", "RSC", "OFA", "FFA-1", "FFA-2", "EBA", "FBA-1", "FBA-2", "Whole Brain",
    ],
}

ROI_LAYOUT = {
    "bs_fz": (1, 2, (6, 4), 1.0),
    "bs_mh": (1, 2, (6, 4), 1.0),
    "tvsd": (1, 3, (6, 6), 0.75),
    "things_eeg1": (2, 4, (6, 4), 0.75),
    "things_eeg2": (2, 4, (6, 4), 0.75),
    "things_meg": (2, 4, (6, 4), 0.75),
    "things_fmri": (7, 4, (6, 4), 0.75),
    "nsd_func1pt8mm_individualROIs": (7, 4, (6, 4), 0.75),
}

EMPTY_PANEL_LEGEND_BENCHMARKS = {"things_eeg1", "things_eeg2", "things_meg"}
OUTSIDE_LEGEND_BENCHMARKS = {"bs_fz", "bs_mh", "tvsd"}


def supervised_spvvs_filter(df: pd.DataFrame, datasets=("imagenet",)) -> pd.Series:
    return (
        (df["backbone_source"] == "spvvs")
        & (df["pretraining_dataset"].isin(datasets))
        & (df["is_adversarially_finetuned"] != True)
        & (df["is_selfsupervised_learning"] != True)
    )


def aggregate_scores(df: pd.DataFrame, groupby_cols, metric=ALIGNMENT_METRIC):
    return apply_hiearchical_aggregation(
        df=df,
        groupby_cols=list(groupby_cols),
        agg_cols=[metric, ACCURACY_METRIC],
        agg_func="mean",
    )


def add_plot_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "benchmark_name" in df.columns:
        df["benchmark_label"] = df["benchmark_name"].map(BENCHMARK_NAME_MAPPING)
    if "roi" in df.columns:
        df["roi"] = df["roi"].replace(ROI_NAME_MAPPING)
    df["pretraining_dataset_label"] = df["pretraining_dataset"].map(DATASET_NAME_MAPPING)
    df["pretraining_dataset_label"] = df["pretraining_dataset_label"].fillna(df["pretraining_dataset"].str.title())
    df["samples_per_class_label"] = df["pretraining_samples_per_class"].replace({0: "full"})
    return df


def pearson_r_p(data: pd.DataFrame, x=ACCURACY_METRIC, y=ALIGNMENT_METRIC):
    data = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data[x].nunique() < 2 or data[y].nunique() < 2:
        return np.nan, np.nan, len(data)
    r, p = stats.pearsonr(data[x], data[y])
    return r, p, len(data)


def format_p(p):
    if pd.isna(p):
        return "nan"
    if p < 1e-3:
        return f"{p:.1e}"
    return f"{p:.3f}"


def format_r_p(data: pd.DataFrame, p_adjusted=np.nan, x=ACCURACY_METRIC, y=ALIGNMENT_METRIC):
    r, p, n = pearson_r_p(data, x=x, y=y)
    if pd.isna(r):
        return f"r=nan, p=nan, n={n}"
    p_to_show = p if pd.isna(p_adjusted) else p_adjusted
    p_label = "p" if pd.isna(p_adjusted) else "p"
    return f"r={r:.2f}, {p_label}={format_p(p_to_show)}, n={n}"


def fdr_bh(p_values):
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid = np.flatnonzero(~pd.isna(p_values))
    if len(valid) == 0:
        return adjusted
    p = np.asarray(p_values, dtype=float)[valid]
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted_valid = np.empty(len(p), dtype=float)
    adjusted_valid[order] = np.minimum(ranked, 1.0)
    adjusted[valid] = adjusted_valid
    return adjusted


def correlation_table(df: pd.DataFrame, groupby_cols, x=ACCURACY_METRIC, y=ALIGNMENT_METRIC):
    rows = []
    if groupby_cols:
        grouped = df.groupby(groupby_cols, dropna=False)
    else:
        grouped = [((), df)]

    for keys, data in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        r, p, n = pearson_r_p(data, x=x, y=y)
        row = {col: key for col, key in zip(groupby_cols, keys)}
        row.update({"r": r, "p": p, "n": n})
        rows.append(row)

    results = pd.DataFrame(rows)
    results["p_fdr_bh"] = fdr_bh(results["p"].to_numpy())
    return results


def adjusted_p_for(results: pd.DataFrame, **selectors):
    matching = results
    for column, value in selectors.items():
        matching = matching[matching[column] == value]
    if len(matching) != 1:
        return np.nan
    return matching["p_fdr_bh"].iloc[0]


def plot_linear_fit(data: pd.DataFrame, ax, color="black", linewidth=1.4, alpha=0.8):
    clean = data[[ACCURACY_METRIC, ALIGNMENT_METRIC]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < 3 or clean[ACCURACY_METRIC].nunique() < 2:
        return
    sns.regplot(
        data=clean,
        x=ACCURACY_METRIC,
        y=ALIGNMENT_METRIC,
        scatter=False,
        ci=None,
        truncate=False,
        color=color,
        line_kws={"linewidth": linewidth, "alpha": alpha},
        ax=ax,
    )


def add_corr_text(ax, data: pd.DataFrame, x=0.96, y=0.04, color="black", fontsize=8, prefix=None, p_adjusted=np.nan):
    text = format_r_p(data, p_adjusted=p_adjusted)
    if prefix:
        text = f"{prefix}: {text}"
    ax.text(
        x,
        y,
        text,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=fontsize,
        color=color,
        bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none", "alpha": 0.72},
    )


def architecture_families(data: pd.DataFrame):
    return [family for family in ARCH_FAMILY_ORDER if family in data["backbone_arch_family"].unique()]


def add_architecture_legend(ax, data: pd.DataFrame, color_by_family=True, loc="upper right", fontsize=7, bbox_to_anchor=None, frameon=False):
    handles = [
        Line2D(
            [], [], linestyle="none", marker=ARCH_FAMILY_MARKERS[family], markersize=LEGEND_MARKER_SIZE,
            markerfacecolor=arch_family_palette[family] if color_by_family else "0.35",
            markeredgecolor="none", label=family,
        )
        for family in architecture_families(data)
    ]
    return ax.legend(
        handles=handles, title="Architecture family", loc=loc, bbox_to_anchor=bbox_to_anchor,
        frameon=frameon, fontsize=fontsize, title_fontsize=fontsize,
    )


df_all["backbone_arch_family"] = df_all["backbone_arch_family"].replace(ARCH_FAMILY_MAPPING)


df_imagenet = df_all[supervised_spvvs_filter(df_all, datasets=("imagenet",))].copy()
df_imagenet = df_imagenet[df_imagenet["benchmark_name"].isin(BENCHMARKS)]
assert len(df_imagenet) > 0, "No supervised SPVVS ImageNet rows found."

df_imagenet[["model_id", "benchmark_name", "roi", ACCURACY_METRIC, ALIGNMENT_METRIC]].head()


# Average Alignment Across Benchmarks

This main figure summarizes the overall relationship by averaging alignment over ROIs and across the eight neural benchmarks for each model. Points show supervised ImageNet- and Ecoset-pretrained SPVVS models across the full validation-accuracy range without fitted trend lines.

In [ ]:
zoom = 0.75
figsize = (8, 6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

# figsize

In [ ]:
df_average_alignment = df_all[supervised_spvvs_filter(df_all, datasets=("imagenet", "ecoset"))].copy()
df_average_alignment = df_average_alignment[df_average_alignment["benchmark_name"].isin(BENCHMARKS)]
df_average_alignment = aggregate_scores(
    df_average_alignment,
    groupby_cols=MODEL_GROUP_COLS,
)
df_average_alignment = add_plot_labels(df_average_alignment)

fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=300)

sns.scatterplot(
    data=df_average_alignment,
    x=ACCURACY_METRIC,
    y=ALIGNMENT_METRIC,
    hue="pretraining_dataset_label",
    hue_order=list(DATASET_PALETTE.keys()),
    palette=DATASET_PALETTE,
    style="backbone_arch_family",
    style_order=architecture_families(df_average_alignment),
    markers=ARCH_FAMILY_MARKERS,
    s=96,
    alpha=0.70,
    linewidth=0,
    legend=False,
    ax=ax,
)

ax.set_title(r"Pretraining: Alignment $\times$ Accuracy", fontsize=PANEL_TITLE_FONTSIZE, fontweight="bold")
ax.set_xlabel("Task Accuracy", fontsize=14, fontweight="bold")
ax.set_ylabel("Average Alignment", fontsize=14, fontweight="bold")
ax.grid(axis="both", linestyle="--", alpha=0.30)

dataset_handles = [
    Line2D([], [], linestyle="none", marker="o", markersize=LEGEND_MARKER_SIZE,
           markerfacecolor=color, markeredgecolor="none", label=label)
    for label, color in DATASET_PALETTE.items()
]
dataset_legend = ax.legend(
    handles=dataset_handles, title="Pretraining dataset", loc="lower right",
    frameon=True, fontsize=14, title_fontsize=14,
)
ax.add_artist(dataset_legend)
add_architecture_legend(
    ax, df_average_alignment, color_by_family=False, loc="upper left", fontsize=14, frameon=True
)

plt.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir_main,
    base_filename="pretraining-accuracy-average_alignment",
    dpi=DPI,
    formats=["pdf", "png", "svg"],
)


**Caption: Average Alignment Across Benchmarks**

\textbf{pretraining-accuracy-average_alignment:} Validation accuracy versus average neural alignment ($r_{\mathrm{NC}}$) across the eight benchmarks for supervised ImageNet- and Ecoset-pretrained \texttt{spvvs} models. Each point represents one model after averaging alignment over ROIs and benchmarks; colors indicate pretraining dataset and markers indicate architecture family.

# Benchmark-Level Overview

Each panel summarizes one benchmark by averaging alignment over its ROIs. Points show supervised ImageNet- and Ecoset-pretrained SPVVS models without fitted trend lines.

In [ ]:
df_benchmark_overview = df_all[supervised_spvvs_filter(df_all, datasets=("imagenet", "ecoset"))].copy()
df_benchmark_overview = df_benchmark_overview[df_benchmark_overview["benchmark_name"].isin(BENCHMARKS)]
df_benchmark_overview = aggregate_scores(
    df_benchmark_overview,
    groupby_cols=MODEL_GROUP_COLS + ["benchmark_name"],
)
df_benchmark_overview = add_plot_labels(df_benchmark_overview)

fig, axes = plt.subplots(2, 4, figsize=(15.0, 7.0), dpi=DPI, sharex=True, squeeze=False)
axes = axes.flatten()

for idx, benchmark_name in enumerate(BENCHMARKS):
    ax = axes[idx]
    data_benchmark = df_benchmark_overview[df_benchmark_overview["benchmark_name"] == benchmark_name]
    sns.scatterplot(
        data=data_benchmark,
        x=ACCURACY_METRIC,
        y=ALIGNMENT_METRIC,
        hue="pretraining_dataset_label",
        hue_order=list(DATASET_PALETTE.keys()),
        palette=DATASET_PALETTE,
        style="backbone_arch_family",
        style_order=architecture_families(df_benchmark_overview),
        markers=ARCH_FAMILY_MARKERS,
        s=58,
        alpha=0.62,
        linewidth=0,
        legend=False,
        ax=ax,
    )
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark_name], fontsize=PANEL_TITLE_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Validation Accuracy" if idx >= 4 else None, fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Pearson r (NC)" if idx % 4 == 0 else None, fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(axis="both", linestyle="--", alpha=0.22)

dataset_handles = [
    Line2D([], [], linestyle="none", marker="o", markersize=LEGEND_MARKER_SIZE,
           markerfacecolor=color, markeredgecolor="none", label=label)
    for label, color in DATASET_PALETTE.items()
]
dataset_legend = axes[6].legend(
    handles=dataset_handles, title="Pretraining dataset", loc="lower right",
    frameon=False, fontsize=ROI_LEGEND_FONTSIZE, title_fontsize=ROI_LEGEND_FONTSIZE,
)
axes[6].add_artist(dataset_legend)
add_architecture_legend(
    axes[-1], df_benchmark_overview, color_by_family=False, loc="lower right", fontsize=ROI_LEGEND_FONTSIZE,
)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename="pretraining-accuracy-benchmark_overview",
    dpi=DPI,
    formats=["pdf", "png", "svg", "jpg"],
)


**Caption: Benchmark-Level Overview**

\textbf{pretraining-accuracy-benchmark_overview:} Validation accuracy versus neural alignment averaged across ROIs for each benchmark. Colors indicate pretraining dataset and markers indicate architecture family.

# ROI Subplots Per Benchmark

Each benchmark gets one figure with a subplot for every ROI. ImageNet and Ecoset SPVVS models are shown in different colors.

In [ ]:
df_roi_compare = df_all[supervised_spvvs_filter(df_all, datasets=("imagenet", "ecoset"))].copy()
df_roi_compare = df_roi_compare[df_roi_compare["benchmark_name"].isin(BENCHMARKS)]

df_roi = aggregate_scores(
    df_roi_compare,
    groupby_cols=MODEL_GROUP_COLS + ["benchmark_name", "roi"],
)
df_roi = df_roi[~((df_roi["benchmark_name"] == "things_fmri") & (df_roi["roi"] == "IT_glasser"))].copy()
df_roi = add_plot_labels(df_roi)

df_roi_stats = correlation_table(
    df_roi,
    groupby_cols=["benchmark_name", "benchmark_label", "roi", "pretraining_dataset_label"],
)
df_roi_stats


In [ ]:
for benchmark_name in BENCHMARKS:
    data_benchmark = df_roi[df_roi["benchmark_name"] == benchmark_name].copy()
    present_rois = data_benchmark["roi"].dropna().unique().tolist()
    rois = [roi for roi in ROI_ORDER[benchmark_name] if roi in present_rois]
    rois.extend(roi for roi in present_rois if roi not in rois)
    if "Whole Brain" in rois:
        rois = [roi for roi in rois if roi != "Whole Brain"] + ["Whole Brain"]
    n_rois = len(rois)
    nrows, ncols, subfigsize, zoom = ROI_LAYOUT[benchmark_name]
    assert n_rois <= nrows * ncols, f"{benchmark_name} has {n_rois} ROIs but only {nrows * ncols} panels."
    figsize = (subfigsize[0] * ncols * zoom, subfigsize[1] * nrows * zoom)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        dpi=DPI,
        sharex=True,
        sharey=False,
        squeeze=False,
    )
    axes = axes.flatten()

    for idx, roi in enumerate(rois):
        ax = axes[idx]
        data_roi = data_benchmark[data_benchmark["roi"] == roi].copy()

        sns.scatterplot(
            data=data_roi,
            x=ACCURACY_METRIC,
            y=ALIGNMENT_METRIC,
            hue="pretraining_dataset_label",
            hue_order=list(DATASET_PALETTE.keys()),
            palette=DATASET_PALETTE,
            style="backbone_arch_family",
            style_order=architecture_families(data_benchmark),
            markers=ARCH_FAMILY_MARKERS,
            s=58,
            alpha=0.58,
            linewidth=0,
            legend=False,
            ax=ax,
        )

        for dataset_label, color in DATASET_PALETTE.items():
            data_dataset = data_roi[data_roi["pretraining_dataset_label"] == dataset_label].copy()
            plot_linear_fit(data_dataset, ax=ax, color=color, linewidth=1.1, alpha=0.85)
            add_corr_text(
                ax,
                data_dataset,
                y=0.04 + STATS_LINE_SPACING * list(DATASET_PALETTE.keys()).index(dataset_label),
                color=color,
                fontsize=ROI_LEGEND_FONTSIZE,
                prefix=dataset_label,
                p_adjusted=adjusted_p_for(
                    df_roi_stats, benchmark_name=benchmark_name, roi=roi,
                    pretraining_dataset_label=dataset_label,
                ),
            )

        ax.set_title(str(roi), fontsize=PANEL_TITLE_FONTSIZE, fontweight="bold")
        ax.set_xlabel("Validation Accuracy" if idx // ncols == nrows - 1 else None, fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
        ax.set_ylabel("Pearson r (NC)" if idx % ncols == 0 else None, fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
        ax.grid(axis="both", linestyle="--", alpha=0.22)

    for ax in axes[n_rois:]:
        ax.axis("off")

    if benchmark_name in EMPTY_PANEL_LEGEND_BENCHMARKS:
        add_architecture_legend(
            axes[n_rois], data_benchmark, color_by_family=False, loc="center left", fontsize=ROI_LEGEND_FONTSIZE,
        )
    elif benchmark_name in OUTSIDE_LEGEND_BENCHMARKS:
        add_architecture_legend(
            axes[n_rois - 1], data_benchmark, color_by_family=False, loc="center left", fontsize=ROI_LEGEND_FONTSIZE,
            bbox_to_anchor=(1.05, 0.5),
        )
    else:
        add_architecture_legend(
            axes[0], data_benchmark, color_by_family=False, loc="lower right", fontsize=ROI_LEGEND_FONTSIZE,
            bbox_to_anchor=(0.98, 0.04 + (len(DATASET_PALETTE) + 1) * STATS_LINE_SPACING),
        )

    fig.suptitle(
        BENCHMARK_NAME_MAPPING[benchmark_name],
        fontsize=SUPTITLE_FONTSIZE,
        fontweight="bold",
        # y=1.01,
    )
    right = 0.86 if benchmark_name in OUTSIDE_LEGEND_BENCHMARKS else 1
    plt.tight_layout(rect=[0, 0, right, 0.97])

    save_figs(
        fig=fig,
        save_dir=save_dir_supp,
        base_filename=f"pretraining-accuracy-rois-{benchmark_name}",
        dpi=DPI,
        formats=["pdf", "png", "svg", "jpg"],
    )


**Captions: Task Accuracy and Neural Alignment within ROIs**

\textbf{pretraining-accuracy-rois-bs_fz:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in FZ-EP for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-bs_mh:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in MH-EP for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-tvsd:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in TVSD-EP for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-things_fmri:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in T-fMRI for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-nsd_func1pt8mm_individualROIs:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in NSD-fMRI for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-things_eeg1:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in T-EEG1 for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-things_eeg2:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in T-EEG2 for supervised ImageNet- and Ecoset-pretrained SPVVS models.

\textbf{pretraining-accuracy-rois-things_meg:} Validation accuracy versus ROI-level neural alignment ($r_{\mathrm{NC}}$) in T-MEG for supervised ImageNet- and Ecoset-pretrained SPVVS models.

## High-Accuracy Model Comparison

Because the full model range suggests a non-linear accuracy--alignment relationship, Figure~\ref{fig:pretraining-accuracy-high_accuracy-imagenet_vs_ecoset} focuses on models with validation accuracy $\geq 0.40$. Here, each model's alignment is averaged across the eight neural benchmarks.

In [ ]:
df_dataset_compare = df_all[supervised_spvvs_filter(df_all, datasets=("imagenet", "ecoset"))].copy()
df_dataset_compare = df_dataset_compare[df_dataset_compare["benchmark_name"].isin(BENCHMARKS)]
df_dataset_compare = df_dataset_compare[df_dataset_compare.task_evaluation_acc >= 0.4].copy()

df_dataset_compare = aggregate_scores(
    df_dataset_compare,
    groupby_cols=MODEL_GROUP_COLS,
)
df_dataset_compare = add_plot_labels(df_dataset_compare)

df_dataset_compare_stats = correlation_table(df_dataset_compare, groupby_cols=["pretraining_dataset_label"])
df_dataset_compare_stats


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.5, 5.5), dpi=DPI)

sns.scatterplot(
    data=df_dataset_compare,
    x=ACCURACY_METRIC,
    y=ALIGNMENT_METRIC,
    hue="pretraining_dataset_label",
    style="backbone_arch_family",
    style_order=architecture_families(df_dataset_compare),
    markers=ARCH_FAMILY_MARKERS,
    palette=DATASET_PALETTE,
    s=96,
    alpha=0.70,
    linewidth=0,
    legend=False,
    ax=ax,
)

for dataset_label, color in DATASET_PALETTE.items():
    data = df_dataset_compare[df_dataset_compare["pretraining_dataset_label"] == dataset_label].copy()
    plot_linear_fit(data, ax=ax, color=color, linewidth=1.8, alpha=0.85)
    add_corr_text(
        ax,
        data,
        y=0.04 + DATASET_COMPARE_STATS_LINE_SPACING * list(DATASET_PALETTE.keys()).index(dataset_label),
        color=color,
        fontsize=LEGEND_FONTSIZE,
        prefix=dataset_label,
        p_adjusted=adjusted_p_for(df_dataset_compare_stats, pretraining_dataset_label=dataset_label),
    )

ax.set_title("High-Accuracy Models: Task Accuracy and Neural Alignment", fontsize=PANEL_TITLE_FONTSIZE, fontweight="bold")
ax.set_xlabel("Validation Accuracy", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
ax.set_ylabel("Pearson r (NC)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
ax.grid(axis="both", linestyle="--", alpha=0.30)
add_architecture_legend(
    ax, df_dataset_compare, color_by_family=False, loc="lower right", fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.98, 0.04 + (len(DATASET_PALETTE) + 1) * DATASET_COMPARE_STATS_LINE_SPACING),
)

plt.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename="pretraining-accuracy-high_accuracy-imagenet_vs_ecoset",
    dpi=DPI,
    formats=["pdf", "png", "svg"],
)

**Caption: Task Accuracy and Neural Alignment among High-Accuracy Models**

\textbf{pretraining-accuracy-high_accuracy-imagenet_vs_ecoset:} Validation accuracy versus mean neural alignment across benchmarks for high-accuracy ImageNet- and Ecoset-pretrained models ($\mathrm{accuracy} \geq 0.40$). Alignment continues to increase with accuracy in this restricted range, although with a shallower trend than in the full accuracy range.

## ROI-Wise Correlation Strength

For each benchmark--ROI pair, the following figures summarize the Pearson correlation between validation accuracy and alignment for ImageNet-pretrained SPVVS models. Error bars are 95\% bootstrap confidence intervals from 1,000 resamples; `Whole Brain` is shown last when available.

In [ ]:
N_BOOTSTRAP = 1000
BOOT_SEED = 0


def bootstrap_r_ci(data, x=ACCURACY_METRIC, y=ALIGNMENT_METRIC, n_boot=N_BOOTSTRAP, seed=BOOT_SEED, alpha=0.05):
    clean = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(clean)
    if n < 3 or clean[x].nunique() < 2 or clean[y].nunique() < 2:
        return np.nan, np.nan, np.nan, n
    xs = clean[x].to_numpy()
    ys = clean[y].to_numpy()
    r = float(np.corrcoef(xs, ys)[0, 1])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    bx = xs[idx]
    by = ys[idx]
    bx_c = bx - bx.mean(axis=1, keepdims=True)
    by_c = by - by.mean(axis=1, keepdims=True)
    num = (bx_c * by_c).sum(axis=1)
    denom = np.sqrt((bx_c ** 2).sum(axis=1) * (by_c ** 2).sum(axis=1))
    with np.errstate(invalid="ignore", divide="ignore"):
        boots = num / denom
    lo = float(np.nanquantile(boots, alpha / 2))
    hi = float(np.nanquantile(boots, 1 - alpha / 2))
    return r, lo, hi, n


def ordered_rois(present_rois, benchmark_name):
    rois = [roi for roi in ROI_ORDER[benchmark_name] if roi in present_rois]
    rois.extend(roi for roi in present_rois if roi not in rois)
    if "Whole Brain" in rois:
        rois = [roi for roi in rois if roi != "Whole Brain"] + ["Whole Brain"]
    return rois


df_hierarchy_src = df_roi[df_roi["pretraining_dataset_label"] == "ImageNet"].copy()

records = []
for benchmark in BENCHMARKS:
    data_bench = df_hierarchy_src[df_hierarchy_src["benchmark_name"] == benchmark]
    present_rois = data_bench["roi"].dropna().unique().tolist()
    rois = ordered_rois(present_rois, benchmark)
    for rank, roi in enumerate(rois):
        data_roi = data_bench[data_bench["roi"] == roi]
        r, lo, hi, n = bootstrap_r_ci(data_roi)
        records.append({
            "benchmark_name": benchmark,
            "benchmark_label": BENCHMARK_NAME_MAPPING[benchmark],
            "roi": roi,
            "roi_rank": rank,
            "r": r,
            "ci_lo": lo,
            "ci_hi": hi,
            "n": n,
        })

df_hier = pd.DataFrame(records)
df_hier

In [ ]:
BAR_WIDTH_PER_ROI = 0.45
BAR_FIG_HEIGHT = 4.0
MIN_PANEL_WIDTH = 3.0

BENCHMARK_GROUPS = [
    ("ep", ["bs_fz", "bs_mh", "tvsd"]),
    ("things_sensor", ["things_eeg1", "things_eeg2", "things_meg"]),
    ("things_fmri", ["things_fmri"]),
    ("nsd", ["nsd_func1pt8mm_individualROIs"]),
]


def plot_roi_bars(ax, data, benchmark, ylim=None):
    n_rois = len(data)
    yerr = np.vstack([data["r"] - data["ci_lo"], data["ci_hi"] - data["r"]])
    color = BENCHMARK_COLORS.get(benchmark, "0.4")
    ax.bar(
        np.arange(n_rois), data["r"].values, yerr=yerr,
        color=color, alpha=0.85, capsize=3, edgecolor="black", linewidth=0.5,
    )
    ax.set_xticks(np.arange(n_rois))
    rotation = 45 if n_rois > 5 else 0
    ha = "right" if rotation else "center"
    ax.set_xticklabels(data["roi"].values, fontsize=10, rotation=rotation, ha=ha)
    ax.set_xlim(-0.6, n_rois - 0.4)
    if ylim is None:
        y_min = float(np.nanmin(data["ci_lo"]))
        y_max = float(np.nanmax(data["ci_hi"]))
        ax.set_ylim(0.99 * y_min, 1.01 * y_max)
    else:
        ax.set_ylim(*ylim)
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=PANEL_TITLE_FONTSIZE, fontweight="bold")
    ax.set_xlabel("ROI", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.3)


for group_name, benchmarks in BENCHMARK_GROUPS:
    group_data = {
        bench: df_hier[df_hier["benchmark_name"] == bench].sort_values("roi_rank")
        for bench in benchmarks
    }
    group_data = {bench: data for bench, data in group_data.items() if not data.empty}
    if not group_data:
        continue

    panel_widths = [
        max(MIN_PANEL_WIDTH, BAR_WIDTH_PER_ROI * len(data) + 1.5)
        for data in group_data.values()
    ]
    fig_width = sum(panel_widths) + 1.0
    sharey = len(group_data) > 1
    fig, axes = plt.subplots(
        1, len(group_data), figsize=(fig_width, BAR_FIG_HEIGHT), dpi=DPI,
        gridspec_kw={"width_ratios": panel_widths}, squeeze=False, sharey=sharey,
    )
    axes = axes.flatten()

    if sharey:
        all_lo = pd.concat([d["ci_lo"] for d in group_data.values()])
        all_hi = pd.concat([d["ci_hi"] for d in group_data.values()])
        shared_ylim = (0.99 * float(np.nanmin(all_lo)), 1.01 * float(np.nanmax(all_hi)))
    else:
        shared_ylim = None

    for idx, (bench, data) in enumerate(group_data.items()):
        plot_roi_bars(axes[idx], data, bench, ylim=shared_ylim)
        if idx == 0:
            axes[idx].set_ylabel(
                "Pearson r\n(accuracy vs. alignment)",
                fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold",
            )

    plt.tight_layout()

    save_figs(
        fig=fig,
        save_dir=save_dir_supp,
        base_filename=f"pretraining-accuracy-roi_corr-{group_name}",
        dpi=DPI,
        formats=["pdf", "png", "svg"],
    )

**Captions: ROI-Wise Correlation Strength**

\textbf{pretraining-accuracy-roi_corr-ep:} ROI-wise accuracy--alignment correlations for FZ-EP, MH-EP, and TVSD-EP in ImageNet-pretrained models; error bars show 95\% bootstrap confidence intervals.

\textbf{pretraining-accuracy-roi_corr-things_sensor:} ROI-wise accuracy--alignment correlations for T-EEG1, T-EEG2, and T-MEG in ImageNet-pretrained models; error bars show 95\% bootstrap confidence intervals.

\textbf{pretraining-accuracy-roi_corr-things_fmri:} ROI-wise accuracy--alignment correlations for T-fMRI in ImageNet-pretrained models; error bars show 95\% bootstrap confidence intervals.

\textbf{pretraining-accuracy-roi_corr-nsd:} ROI-wise accuracy--alignment correlations for NSD-fMRI in ImageNet-pretrained models; error bars show 95\% bootstrap confidence intervals.

## Overall Analysis

Figure~\ref{fig:pretraining-accuracy-average_alignment} asks whether the task competence acquired during supervised visual pretraining predicts a model's ability to account for neural responses beyond any single dataset, modality, or region of interest. Averaging $r_{\mathrm{NC}}$ over ROIs and across the eight benchmarks produces a conservative model-level summary: an effect visible here cannot be attributed to an isolated ROI or to an unusually favorable benchmark. The resulting organization is strikingly consistent across ImageNet- and Ecoset-pretrained models: models with weak validation performance also provide weak average neural predictions, whereas progressively more accurate models occupy successively higher alignment regimes. This cross-pretraining-dataset agreement suggests that the association is not specific to the ImageNet label space, but instead reflects representational properties that accompany successful supervised object recognition across distinct natural-image training distributions. At the same time, the visual compression of alignment among the highest-accuracy models is consistent with diminishing returns: improvements sufficient to separate poorly performing from competent recognition systems correspond to large gains in neural alignment, while further gains among already strong systems appear to yield smaller changes in the average benchmark score. Importantly, this figure establishes a population-level association rather than a causal effect of accuracy or a ranking of architectures: validation accuracy, architecture family, capacity, and other pretraining choices vary together, and benchmark averaging can conceal systematic regional differences. The benchmark- and ROI-resolved appendix analyses therefore serve as essential checks on whether this global pattern is broadly distributed and where its correspondence to neural responses is strongest.

## Appendix Analysis

We first examine the relationship between supervised pretraining accuracy and neural alignment at the benchmark level. Figure~\ref{fig:pretraining-accuracy-benchmark_overview} plots alignment averaged across ROIs for each of the eight benchmarks, showing ImageNet- and Ecoset-pretrained models from multiple architecture families without imposing a fitted trend. Across benchmarks, models with higher validation accuracy tend to obtain higher neural alignment, motivating a more detailed examination of the regional structure of this association.

We next resolve the relationship within individual ROIs. Figures~\ref{fig:pretraining-accuracy-rois-bs_fz}, \ref{fig:pretraining-accuracy-rois-bs_mh}, and \ref{fig:pretraining-accuracy-rois-tvsd} show positive relationships in FZ-EP, MH-EP, and TVSD-EP, with stronger accuracy--alignment coupling in higher-level visual ROIs. For ImageNet-pretrained models, correlations increase from V1 to V2 in FZ-EP ($r=0.376$ to $0.533$), from V4 to IT in MH-EP ($r=0.700$ to $0.831$), and from V1 to V4 to IT in TVSD-EP ($r=0.616$, $0.776$, and $0.820$); Ecoset-pretrained models show the same ordering. Figures~\ref{fig:pretraining-accuracy-rois-things_fmri} and \ref{fig:pretraining-accuracy-rois-nsd_func1pt8mm_individualROIs} extend the positive relationship across the ROI sets of T-fMRI and NSD-fMRI, and Figures~\ref{fig:pretraining-accuracy-rois-things_eeg1}, \ref{fig:pretraining-accuracy-rois-things_eeg2}, and \ref{fig:pretraining-accuracy-rois-things_meg} show corresponding positive associations across T-EEG1, T-EEG2, and T-MEG sensor-region groupings. Across the 84 ROIs evaluated for each pretraining dataset, all 168 ImageNet- or Ecoset-specific correlations are positive and significant after Benjamini--Hochberg correction ($p_{\mathrm{FDR}} < 10^{-11}$).

The full accuracy range suggests a non-linear relationship: alignment increases rapidly at low accuracy levels and appears to increase with a shallower slope among higher-accuracy models. Figure~\ref{fig:pretraining-accuracy-high_accuracy-imagenet_vs_ecoset} isolates models with validation accuracy $\geq 0.40$ and averages alignment across all eight benchmarks. Even within this range, the direction remains strongly positive for both Ecoset-pretrained models ($r=0.72$, $p_{\mathrm{FDR}}=5.8\times10^{-21}$, $n=124$) and ImageNet-pretrained models ($r=0.67$, $p_{\mathrm{FDR}}=3.7\times10^{-20}$, $n=142$). Thus, although the full data suggest diminishing gains in alignment as accuracy increases, alignment continues to increase with task performance among high-accuracy models.

Finally, Figures~\ref{fig:pretraining-accuracy-roi_corr-ep}, \ref{fig:pretraining-accuracy-roi_corr-things_sensor}, \ref{fig:pretraining-accuracy-roi_corr-things_fmri}, and \ref{fig:pretraining-accuracy-roi_corr-nsd} directly summarize ROI-wise correlation strengths for ImageNet-pretrained models with bootstrap confidence intervals. In FZ-EP, MH-EP, and TVSD-EP, correlations range from $r=0.38$ to $0.83$ and visibly reinforce the stronger coupling in higher-level visual ROIs. In the T-EEG1, T-EEG2, and T-MEG benchmarks, they range from $r=0.65$ to $0.86$. T-fMRI and NSD-fMRI also show broadly strong effects across ROIs, with ranges of $r=0.68$--$0.93$ and $r=0.69$--$0.87$, respectively. Taken together, these analyses show that higher supervised pretraining task accuracy is a reliable correlate of stronger neural alignment across the evaluated model collection, with particularly strong correspondence in higher-level visual ROIs. Because accuracy also co-varies with architecture family and other pretraining choices, these results characterize an association rather than a causal effect of accuracy alone.